# BandList.fit — spectral parameter recovery test

In [1]:
%run pylib/tools dark
from utilities.ipynb_docgen import show, show_date
from pylib.tools import np
from importlib import reload
from pylib.psf_func import PSFlist
from like3 import bands as bands_mod; reload(bands_mod)
BandList = bands_mod.BandList
from like3 import sourcelist as sourcelist_mod; reload(sourcelist_mod)
SourceModel = sourcelist_mod.SourceModel

show_date()

<h5 style="text-align:right; margin-right:15px"> 2026-04-15 08:35</h5>

## Setup: simulate data with known parameters

In [3]:
reload(bands_mod)
BandList = bands_mod.BandList
# Build source model.
source_model = SourceModel.demo()

df = PSFlist.demo_df()
df['nside'] = BandList.nside

# Estimate per-source pivot energies from a temporary fitted view,
# then apply them to the real model before running the fit tests.
pivot_by_name = {}
with source_model.view() as trial_sources:
    trial_bandlist = BandList(trial_sources, df)
    trial_bandlist.simulate(random_state=42)
    trial_bandlist.fit(method='l-bfgs-b', use_gradient=True, quiet=True)

    # Reconstruct covariance from fit diagnostics and attach it to models.
    corr = np.asarray(trial_bandlist.fit_info['correlation'], dtype=float)
    sig = np.asarray(trial_bandlist.fit_info['errors'], dtype=float)
    cov = corr * np.outer(sig, sig)
    trial_sources.parameters.set_covariance(cov)

    for src in trial_sources:
        model = src.model
        pivot = np.nan
        if hasattr(model, 'pivot_energy'):
            try:
                pivot = float(model.pivot_energy(exception=False))
            except Exception:
                pivot = np.nan
        pivot_by_name[src.name] = pivot

pivot_rows = []
for src in source_model:
    model = src.model
    e0_old = float(getattr(model, 'e0', np.nan))
    pivot = float(pivot_by_name.get(src.name, np.nan))

    if hasattr(model, 'set_e0') and np.isfinite(pivot) and pivot > 0:
        model.set_e0(pivot)

    e0_new = float(getattr(model, 'e0', np.nan))
    pivot_rows.append((src.name, e0_old, pivot, e0_new))

show("""
**e0 alignment to pivot energy**

| Source | Old e0 (MeV) | Pivot (MeV) | New e0 (MeV) |
|--------|:------------:|:-----------:|:------------:|
""" + "\n".join(
    f"| {name} | {old:.1f} | {pv:.1f} | {new:.1f} |"
    for name, old, pv, new in pivot_rows
))

bandlist = BandList(source_model, df)
bandlist.simulate(random_state=42)

# Record the true parameter values before any perturbation
true_params = source_model.parameters.get_parameters().copy()
true_names  = source_model.parameter_names
show(f'**True parameters:** {dict(zip(true_names, true_params.round(4)))}')


Model: SourceModel: 2 sources with 4 free parameters


AttributeError: type object 'PixelTable' has no attribute 'nside'

## Fit: perturb parameters then recover with BandList.fit()

In [ ]:
# Perturb all free parameters by 20% to give the fitter something to recover from
perturbed = true_params * 1.2
source_model.parameters.set_parameters(perturbed)
show(f'**Perturbed parameters:** {dict(zip(true_names, perturbed.round(4)))}')

# Run the fit with l-bfgs-b and analytic gradient
fitvalue, best_pars, errors = bandlist.fit(method='l-bfgs-b', use_gradient=True, quiet=False)

show(f"""
**Fit result** (log-likelihood  change = {fitvalue:.1f})

| Parameter | True | Recovered | Error | Recovered/True-Error |
|-----------|:----:|:---------:|:-----:|:--------------------:|
""" + "\n".join(
    f"| {n} | {t:.4f} | {r:.4f} | {e:.4f} | {(r - t) / e if e > 0 else np.nan:.1f} |"
    for n, t, r, e in zip(true_names, true_params, best_pars, errors)
))


**Perturbed parameters:** {'Pulsar_Norm': -12.1101, 'Pulsar_Index': 1.694, 'Blazar_Norm': -12.5282, 'Blazar_Index': 2.4}

using optimize.fmin_l_bfgs_b with parameter bounds [[-15.  -3.]
 [ -5.   5.]
 [-17.  -3.]
 [ -5.   5.]]
, kw= {}
{'grad': array([-0.17591011,  0.00911366,  0.25549863, -0.08313293]), 'task': 'CONVERGENCE: REL_REDUCTION_OF_F_<=_FACTR*EPSMCH', 'funcalls': 19, 'nit': 13, 'warnflag': 0}
Function value at minimum: -82994.383
Attempting to invert full hessian...


**Fit result** (log-likelihood  change = -82994.4)

| Parameter | True | Recovered | Error | Recovered/True-Error |
|-----------|:----:|:---------:|:-----:|:--------------------:|
| Pulsar_Norm | -10.0917 | -10.0842 | 0.0035 | 2.1 |
| Pulsar_Index | 1.4117 | 1.3959 | 0.0089 | -1.8 |
| Blazar_Norm | -10.4402 | -10.4323 | 0.0061 | 1.3 |
| Blazar_Index | 2.0000 | 1.9412 | 0.0144 | -4.1 |

## Band selection: fit using only a subset of bands

In [ ]:
# Perturb again
source_model.parameters.set_parameters(true_params * 1.2)

# Fit using only the 6 middle bands (indices 3-8)
mid_bands = list(range(3, 9))
bandlist.select(mid_bands)
show(f'Fitting with bands {mid_bands}: {[bandlist[i] for i in mid_bands]}')

fitvalue_mid, best_pars_mid, errors_mid = bandlist.fit(method='l-bfgs-b', use_gradient=True, quiet=True)

# Reset to all bands
bandlist.select()

show(f"""
**Subset-band fit result** (negative log-likelihood = {fitvalue_mid:.1f})

| Parameter | True | Recovered | Error | Recovered/True-Error |
|-----------|:----:|:---------:|:-----:|:--------------------:|
""" + "\n".join(
    f"| {n} | {t:.4f} | {r:.4f} | {e:.4f} | {(r - t) / e if e > 0 else np.nan:.1f} |"
    for n, t, r, e in zip(true_names, true_params, best_pars_mid, errors_mid)
))


Fitting with bands [3, 4, 5, 6, 7, 8]: [Band( FRONT, 0.75 GeV, nside 128), Band( FRONT, 1.33 GeV, nside 256), Band( FRONT, 2.37 GeV, nside 512), Band( FRONT, 4.22 GeV, nside 512), Band( FRONT, 7.50 GeV, nside 512), Band( FRONT, 13.34 GeV, nside 1024)]

**Subset-band fit result** (negative log-likelihood = -26266.2)

| Parameter | True | Recovered | Error | Recovered/True-Error |
|-----------|:----:|:---------:|:-----:|:--------------------:|
| Pulsar_Norm | -10.0917 | -10.0939 | 0.0126 | -0.2 |
| Pulsar_Index | 1.4117 | 1.3733 | 0.0243 | -1.6 |
| Blazar_Norm | -10.4402 | -10.4044 | 0.0277 | 1.3 |
| Blazar_Index | 2.0000 | 1.9824 | 0.0398 | -0.4 |

In [ ]:
from time import perf_counter
import numpy as np

# Compare fit runtime with and without analytic gradient from the same start point.
def timed_fit(use_gradient, start_pars):
    source_model.parameters.set_parameters(start_pars.copy())
    t0 = perf_counter()
    out = bandlist.fit(method='l-bfgs-b', use_gradient=use_gradient, quiet=True)
    dt = perf_counter() - t0
    return dt, out

n_trials = 3
start_pars = true_params * 1.2

# Ensure all bands are active for a fair comparison.
bandlist.select()

times_no_grad = []
times_with_grad = []

for _ in range(n_trials):
    dt, _ = timed_fit(False, start_pars)
    times_no_grad.append(dt)

for _ in range(n_trials):
    dt, _ = timed_fit(True, start_pars)
    times_with_grad.append(dt)

mean_no_grad = float(np.mean(times_no_grad))
mean_with_grad = float(np.mean(times_with_grad))
speedup = mean_no_grad / mean_with_grad if mean_with_grad > 0 else np.nan

print('BandList.fit timing comparison (method=l-bfgs-b)')
print(f'  trials: {n_trials}')
print(f'  no gradient : {mean_no_grad:.3f} s  (runs={np.round(times_no_grad, 3)})')
print(f'  with gradient: {mean_with_grad:.3f} s  (runs={np.round(times_with_grad, 3)})')
print(f'  speedup (no_grad / grad): {speedup:.1f}x')

BandList.fit timing comparison (method=l-bfgs-b)
  trials: 3
  no gradient : 16.455 s  (runs=[14.679 14.924 19.763])
  with gradient: 5.328 s  (runs=[5.333 5.771 4.878])
  speedup (no_grad / grad): 3.1x


In [ ]:
from time import perf_counter
import numpy as np

bandlist.select()  # all bands

configs = [
    ('simplex',  False),
    ('powell',   False),
    ('l-bfgs-b', False),
    ('l-bfgs-b', True),
]
n_reps = 2  # repeat each to reduce noise

rows = []
for method, use_grad in configs:
    label = f'{method}{"  +grad" if use_grad else ""}'
    times = []
    for _ in range(n_reps):
        source_model.parameters.set_parameters(true_params * 1.2)
        t0 = perf_counter()
        # estimate_errors=False: only measure optimizer time, not Hessian cost
        fval, pars = bandlist.fit(method=method, use_gradient=use_grad,
                                  quiet=True, estimate_errors=False)
        times.append(perf_counter() - t0)
    mean_t = float(np.mean(times))
    residuals = np.abs(pars - true_params)
    rows.append((label, mean_t, float(residuals.max())))

# Sort fastest first
rows.sort(key=lambda x: x[1])
fastest = rows[0][1]


In [ ]:

show(rf"""
#### Method speed comparison ({n_reps} reps each, all 12 bands)

| Method | Time (s) | Rel time | Max | 
|--------|:--------:|:--------:|:----------:|
""" + "\n".join(
    f"| {label} | {t:.2f} | {t/fastest:.1f}x | {maxd:.3f} |"
    for label, t, maxd in rows
))

#### Method speed comparison (2 reps each, all 12 bands)

| Method | Time (s) | Rel time | Max | 
|--------|:--------:|:--------:|:----------:|
| l-bfgs-b  +grad | 2.06 | 1.0x | 0.059 |
| powell | 3.49 | 1.7x | 0.061 |
| simplex | 13.81 | 6.7x | 0.059 |
| l-bfgs-b | 14.79 | 7.2x | 0.059 |

## Variance vs predicted errors: ensemble of simulations

In [ ]:
import contextlib
from tqdm.auto import tqdm
import joblib
from joblib import Parallel, delayed

@contextlib.contextmanager
def tqdm_joblib(tqdm_object):
    """Patch joblib to report progress into a tqdm bar."""
    class TqdmBatchCompletionCallback(joblib.parallel.BatchCompletionCallBack):
        def __call__(self, *args, **kwargs):
            tqdm_object.update(n=self.batch_size)
            return super().__call__(*args, **kwargs)
    old = joblib.parallel.BatchCompletionCallBack
    joblib.parallel.BatchCompletionCallBack = TqdmBatchCompletionCallback
    try:
        yield tqdm_object
    finally:
        joblib.parallel.BatchCompletionCallBack = old
        tqdm_object.close()

def _run_one(seed):
    source_model.parameters.set_parameters(true_params.copy())
    bandlist.simulate(random_state=seed)
    source_model.parameters.set_parameters(true_params.copy())
    _, best_sim, err_sim = bandlist.fit(method='l-bfgs-b', use_gradient=True, quiet=True)
    return best_sim, err_sim

n_sim = 200
with tqdm_joblib(tqdm(total=n_sim)):
    results = Parallel(n_jobs=-1)(delayed(_run_one)(seed) for seed in range(n_sim))

recovered, predicted_errors = map(np.array, zip(*results))

obs_std   = recovered.std(axis=0)
pred_mean = predicted_errors.mean(axis=0)
bias      = recovered.mean(axis=0) - true_params

# Coverage: if errors are well calibrated, expect ~68% and ~95% within 1σ and 2σ.
z = (recovered - true_params[None, :]) / np.where(predicted_errors > 0, predicted_errors, np.nan)
cover_1s = np.nanmean(np.abs(z) <= 1.0, axis=0)
cover_2s = np.nanmean(np.abs(z) <= 2.0, axis=0)

show(f"""
## Ensemble fit validation  (n = {n_sim} simulations)

| Parameter | True | Bias | Obs σ | Predicted σ | Obs/Pred | 68% Cov | 95% Cov |
|-----------|:----:|:----:|:-----:|:-----------:|:--------:|:-------:|:-------:|
""" + "\n".join(
    f"| {n} | {t:.4f} | {b:.4f} | {o:.4f} | {p:.4f} | {o/p:.3f} | {c1:.3f} | {c2:.3f} |"
    for n, t, b, o, p, c1, c2 in zip(true_names, true_params, bias, obs_std, pred_mean, cover_1s, cover_2s)
))


  0%|          | 0/200 [00:00<?, ?it/s]

## Ensemble fit validation  (n = 200 simulations)

| Parameter | True | Bias | Obs σ | Predicted σ | Obs/Pred | 68% Cov | 95% Cov |
|-----------|:----:|:----:|:-----:|:-----------:|:--------:|:-------:|:-------:|
| Pulsar_Norm | -10.0917 | 0.0046 | 0.0080 | 0.0036 | 2.254 | 0.335 | 0.545 |
| Pulsar_Index | 1.4117 | -0.0146 | 0.0080 | 0.0089 | 0.897 | 0.250 | 0.700 |
| Blazar_Norm | -10.4402 | 0.0119 | 0.0076 | 0.0061 | 1.250 | 0.200 | 0.510 |
| Blazar_Index | 2.0000 | -0.0604 | 0.0154 | 0.0143 | 1.078 | 0.000 | 0.020 |

## Parameter degeneracy check from ensemble fits

In [ ]:
# Correlation matrix from recovered parameter ensemble
corr = np.corrcoef(recovered, rowvar=False)

# Pull out the blazar parameter relationship explicitly
name_to_idx = {n: i for i, n in enumerate(true_names)}
ibn = name_to_idx.get('Blazar_Norm')
ibi = name_to_idx.get('Blazar_Index')
blazar_corr = corr[ibn, ibi] if ibn is not None and ibi is not None else np.nan

# Report strongest off-diagonal absolute correlations
pairs = []
for i in range(len(true_names)):
    for j in range(i + 1, len(true_names)):
        pairs.append((abs(corr[i, j]), true_names[i], true_names[j], corr[i, j]))
pairs.sort(reverse=True)

show(f"""
## Ensemble correlation diagnostics

Blazar_Norm vs Blazar_Index correlation: **{blazar_corr:.3f}**

| Pair | Corr |
|------|:----:|
""" + "\n".join(
    f"| {a} vs {b} | {c:+.3f} |"
    for _, a, b, c in pairs[:6]
))

## Ensemble correlation diagnostics

Blazar_Norm vs Blazar_Index correlation: **-0.023**

| Pair | Corr |
|------|:----:|
| Pulsar_Index vs Blazar_Norm | -0.572 |
| Pulsar_Norm vs Pulsar_Index | -0.483 |
| Pulsar_Index vs Blazar_Index | -0.114 |
| Pulsar_Norm vs Blazar_Norm | +0.100 |
| Blazar_Norm vs Blazar_Index | -0.023 |
| Pulsar_Norm vs Blazar_Index | -0.017 |

## Error calibration from ensemble coverage

In [ ]:
# Calibrate predicted parameter errors using empirical |z| quantiles.
# z is already defined in the ensemble cell as (recovered - true) / predicted_error.

abs_z = np.abs(z)

# Per-parameter scales to match nominal Normal thresholds.
q68 = np.nanquantile(abs_z, 0.6827, axis=0)   # target ~1.0
q95 = np.nanquantile(abs_z, 0.9545, axis=0)   # target ~1.96
scale_68 = q68
scale_95 = q95 / 1.96
scale_reco = np.maximum(scale_68, scale_95)

# Global scale as a simpler one-number fallback.
global_scale = float(np.nanmedian(scale_reco))

# Coverage after applying per-parameter recommended scaling.
z_adj = z / scale_reco[None, :]
cov68_adj = np.nanmean(np.abs(z_adj) <= 1.0, axis=0)
cov95_adj = np.nanmean(np.abs(z_adj) <= 2.0, axis=0)

show(f"""
## Error inflation calibration (from ensemble)

Recommended global scale (median over parameters): **{global_scale:.3f}**

| Parameter | Scale(68%) | Scale(95%) | Recommended scale | 68% Cov (adj) | 95% Cov (adj) |
|-----------|:----------:|:----------:|:-----------------:|:-------------:|:-------------:|
""" + "\n".join(
    f"| {name} | {s68:.3f} | {s95:.3f} | {sr:.3f} | {c68:.3f} | {c95:.3f} |"
    for name, s68, s95, sr, c68, c95 in zip(true_names, scale_68, scale_95, scale_reco, cov68_adj, cov95_adj)
))

## Error inflation calibration (from ensemble)

Recommended global scale (median over parameters): **2.558**

| Parameter | Scale(68%) | Scale(95%) | Recommended scale | 68% Cov (adj) | 95% Cov (adj) |
|-----------|:----------:|:----------:|:-----------------:|:-------------:|:-------------:|
| Pulsar_Norm | 2.628 | 2.543 | 2.628 | 0.680 | 0.955 |
| Pulsar_Index | 1.947 | 1.672 | 1.947 | 0.680 | 0.985 |
| Blazar_Norm | 2.489 | 2.095 | 2.489 | 0.680 | 0.995 |
| Blazar_Index | 4.755 | 3.182 | 4.755 | 0.680 | 1.000 |